# Proyecto integrador de Ciencia de Datos e Inteligencia Artificial

**Fase 4: dashboard interactivo, informe final y sustentación**  
**Estudiante:** Jennifer Juliana Bayona Rojas  
**Programa:** Maestría en Inteligencia Artificial · Universidad de La Salle

Este notebook convierte los resultados de Fase 3 en una aplicación Streamlit y explica
la publicación en GitHub + Streamlit Community Cloud. Cada bloque de explicación
precede al código que debes ejecutar. No necesitas volver a entrenar los modelos.

**Cómo empezar:** abre este archivo en Colab o Jupyter y ejecuta las celdas en orden.
Si usas el paquete completo, conserva su estructura de carpetas. Si abres solamente
este notebook en Colab, ten disponible `artefactos_fase3.zip` para subirlo en el paso 2.

El notebook prepara y verifica archivos. La publicación se realiza en el paso 12
con tu cuenta de GitHub y Streamlit. **Un notebook ejecutado no equivale a una app publicada.**

## Ruta de trabajo

| Paso | Resultado |
|---|---|
| 1 | Entorno con versiones fijadas |
| 2 | Carga del ZIP o reconocimiento del proyecto completo |
| 3 | Inspección de pedidos y métricas |
| 4 | Funciones de datos, filtros e indicadores |
| 5 | Inferencia con pipelines de Fase 3 |
| 6 | Prueba de predicción de un pedido |
| 7 | Aplicación Streamlit con cuatro vistas |
| 8 | Dependencias y configuración de publicación |
| 9 | Reproducción de métricas y prueba de la aplicación |
| 10 | Informe, narrativa y guion de sustentación |
| 11 | Ejecución local y descarga del proyecto |
| 12 | Publicación en GitHub y Streamlit Community Cloud |
| 13 | Registro de enlaces y verificación final |

La guía pide una aplicación web funcional y desplegada, filtros, tarjetas KPI,
predicción en vivo, mapa de clústeres, informe formal y repositorio reproducible.

## 1. Preparar el entorno

Conservamos `scikit-learn==1.6.1`, la versión utilizada para guardar tus modelos.
El resto de las versiones corresponde al entorno de inferencia comprobado para esta
entrega. No copiamos todas las dependencias de entrenamiento al dashboard: los
modelos elegidos para la app no requieren PyTorch.

El despliegue se recomienda con **Python 3.12**. Si ya importaste pandas, NumPy,
scikit-learn o Streamlit antes de cambiar sus versiones, reinicia la sesión de Colab
y vuelve a ejecutar desde aquí cuando la celda lo indique.

In [1]:
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import sys
import subprocess
import json
import shutil
import zipfile
import importlib

VERSIONES = {
    "streamlit": "1.51.0", "plotly": "6.5.0", "pandas": "2.2.3",
    "numpy": "2.3.5", "scipy": "1.16.3", "scikit-learn": "1.6.1", "joblib": "1.5.3",
}
pendientes = []
for paquete, requerida in VERSIONES.items():
    try:
        if version(paquete) != requerida:
            pendientes.append(f"{paquete}=={requerida}")
    except PackageNotFoundError:
        pendientes.append(f"{paquete}=={requerida}")
modulos_cargados = any(x in sys.modules for x in ["numpy", "pandas", "sklearn", "streamlit", "scipy"])
if pendientes:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *pendientes])
    importlib.invalidate_caches()
    if modulos_cargados:
        raise RuntimeError("Instalación terminada. Reinicia la sesión/kernel y ejecuta desde el paso 1.")
print("Python:", sys.version.split()[0])
print("Entorno preparado:", VERSIONES)

Python: 3.12.14
Entorno preparado: {'streamlit': '1.51.0', 'plotly': '6.5.0', 'pandas': '2.2.3', 'numpy': '2.3.5', 'scipy': '1.16.3', 'scikit-learn': '1.6.1', 'joblib': '1.5.3'}


## 2. Cargar y organizar los archivos

**Opción A — paquete completo:** el código reconoce `data/` y `models/`; no subas
de nuevo el ZIP original.

**Opción B — notebook solo en Colab:** cuando aparezca el selector, sube
`artefactos_fase3.zip`. También puedes subir `base_depurada_fase1.csv` y los tres
notebooks anteriores para que queden incluidos en el repositorio generado.

En Jupyter local, coloca el ZIP junto al notebook. Si la ubicación es diferente,
ajusta `CARPETA_ENTRADA`. Los originales se copian, no se sobrescriben.

In [2]:
CARPETA_ENTRADA = Path.cwd()
candidatos = [CARPETA_ENTRADA, CARPETA_ENTRADA.parent, CARPETA_ENTRADA / "proyecto_fase4"]
PROYECTO = next((p for p in candidatos if (p / "data/dataset_procesado.csv").exists()),
                CARPETA_ENTRADA / "proyecto_fase4")
for carpeta in ["data", "models", "reports", "src", "dashboard", "notebooks"]:
    (PROYECTO / carpeta).mkdir(parents=True, exist_ok=True)

if not (PROYECTO / "data/dataset_procesado.csv").exists():
    ZIP_FASE3 = CARPETA_ENTRADA / "artefactos_fase3.zip"
    if not ZIP_FASE3.exists():
        try:
            from google.colab import files
        except ImportError:
            raise FileNotFoundError("Coloca artefactos_fase3.zip en CARPETA_ENTRADA y repite esta celda.")
        subidos = files.upload()
        if "artefactos_fase3.zip" not in subidos:
            raise FileNotFoundError("Debes subir el archivo artefactos_fase3.zip.")
    carpeta_extraida = PROYECTO.parent / "entrada_fase3"
    carpeta_extraida.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZIP_FASE3) as archivo_zip:
        for item in archivo_zip.infolist():
            if not (carpeta_extraida / item.filename).resolve().is_relative_to(carpeta_extraida.resolve()):
                raise ValueError("El ZIP contiene una ruta fuera de la carpeta esperada.")
        archivo_zip.extractall(carpeta_extraida)
    origen = carpeta_extraida / "artefactos_fase3"
    for f in origen.iterdir():
        if not f.is_file():
            continue
        if f.name in ["dataset_procesado.csv", "segmentos_kmeans.csv"]:
            destino = PROYECTO / "data" / f.name
        elif f.suffix in [".joblib", ".pt"]:
            destino = PROYECTO / "models" / f.name
        elif f.name == "requirements.txt":
            destino = PROYECTO / "reports/requirements_fase3_original.txt"
        else:
            destino = PROYECTO / "reports" / f.name
        shutil.copy2(f, destino)

# Incluye antecedentes si los colocaste junto al notebook o dentro de upload/.
for carpeta in [CARPETA_ENTRADA, CARPETA_ENTRADA / "upload"]:
    if not carpeta.exists():
        continue
    for f in carpeta.iterdir():
        if f.name == "base_depurada_fase1.csv":
            destino = PROYECTO / "data" / f.name
        elif f.suffix == ".ipynb" and f.name.startswith(("Fase_1_", "Fase_2_", "Fase_3_", "Fase_4_")):
            destino = PROYECTO / "notebooks" / f.name
        else:
            continue
        if destino.resolve() != f.resolve():
            shutil.copy2(f, destino)
print("Proyecto preparado en:", PROYECTO.resolve())

Proyecto preparado en: /workspace/scratch/95ce581e477a/proyecto_fase4


## 3. Comprobar el contenido de Fase 3

Esperamos 138.116 pedidos, de los cuales 82.600 están en USD. Las tablas de métricas
son resultados ya evaluados; no los recalculamos para cada filtro del dashboard.

**Decisiones de comunicación:** importes separados por moneda; venta y ticket solo
de pedidos completados; tasa de devolución sobre completados y devueltos; clasificación
exploratoria, porque su precisión en prueba es 8,38 % y su AUC es 0,5605.

In [3]:
import pandas as pd
import numpy as np
from IPython.display import display

datos_originales = pd.read_csv(PROYECTO / "data/dataset_procesado.csv", parse_dates=["order_date"])
assert len(datos_originales) == 138116
assert datos_originales["order_id"].is_unique
print("Pedidos:", len(datos_originales))
print("Periodo:", datos_originales.order_date.min().date(), "a", datos_originales.order_date.max().date())
display(datos_originales["currency"].value_counts().to_frame("Pedidos"))
display(pd.read_csv(PROYECTO / "reports/metricas_regresion.csv", index_col=0))
display(pd.read_csv(PROYECTO / "reports/metricas_clasificacion.csv", index_col=0))
display(pd.read_csv(PROYECTO / "reports/metricas_clustering.csv"))

Pedidos: 138116
Periodo: 2021-01-01 a 2025-12-31


,Pedidos
currency,
USD,82600
GBP,20235
EUR,11593
CAD,7015
AUD,6794
INR,5707
AED,4172


,MAE,RMSE,R2
Entrenamiento,613.295861,889.517721,0.383506
Validación,609.484397,880.403130,0.371968
Prueba,608.749197,885.115930,0.382417


,Accuracy,Precision,Recall,F1-Score,AUC-ROC,Average Precision
Validación,0.336476,0.085471,0.786469,0.154186,0.557369,0.091243
Prueba,0.330010,0.083815,0.776056,0.151290,0.560520,0.099920


,Modelo,Configuración,Clústeres,Registros,Silueta,Ruido (%),Registros para silueta
0,K-Means,K=4,4,82600,0.302757,0.00,7932
1,DBSCAN,"eps=0.481, min_samples=20",3,8000,0.208138,0.85,7932


## 4. Crear las funciones de datos y KPI

Este archivo valida la unión con los clústeres mediante `indice_origen` y verifica
edad, cantidad e importe. Así evitamos unir por posición dos tablas de distinto tamaño.

Un filtro de categorías vacío produce cero registros. Si no hay pedidos finalizados,
la tasa se muestra como «sin datos». Los KPI rechazan la suma de monedas diferentes.

In [4]:
ruta = PROYECTO / "src/__init__.py"
contenido = r'''"""Funciones del dashboard del proyecto integrador."""
'''
ruta.parent.mkdir(parents=True, exist_ok=True)
ruta.write_text(contenido, encoding="utf-8")
print("Creado:", ruta)

Creado: /workspace/scratch/95ce581e477a/proyecto_fase4/src/__init__.py


In [5]:
ruta = PROYECTO / "src/datos.py"
contenido = r'''"""Carga y segmentación descriptiva de pedidos, sin volver a ajustar modelos."""
from pathlib import Path
import json
import numpy as np
import pandas as pd


def cargar_datos(raiz):
    raiz = Path(raiz)
    df = pd.read_csv(raiz / "data/dataset_procesado.csv", parse_dates=["order_date"])
    segmentos = pd.read_csv(raiz / "data/segmentos_kmeans.csv")
    if df["indice_origen"].duplicated().any() or segmentos["indice_origen"].duplicated().any():
        raise ValueError("indice_origen debe ser único para unir los pedidos con sus clústeres.")
    if df["order_id"].duplicated().any() or df["order_date"].isna().any():
        raise ValueError("Se encontraron pedidos duplicados o fechas inválidas.")
    usd = df.loc[df["currency"].eq("USD")].set_index("indice_origen")
    seg = segmentos.set_index("indice_origen")
    if set(usd.index) != set(seg.index):
        raise ValueError("Los segmentos deben corresponder exactamente a los pedidos en USD.")
    for col in ["customer_age", "quantity", "net_sales"]:
        if not np.allclose(usd.loc[seg.index, col], seg[col], rtol=1e-9, atol=1e-9):
            raise ValueError(f"La unión de segmentos no coincide para {col}.")
    columnas = ["indice_origen", "cluster_kmeans", "perfil_comercial"]
    df = df.merge(segmentos[columnas], on="indice_origen", how="left", validate="one_to_one")
    if df.loc[df.currency.ne("USD"), "cluster_kmeans"].notna().any():
        raise ValueError("Se asignaron clústeres a una moneda diferente de USD.")
    df["cluster_kmeans"] = df["cluster_kmeans"].astype("Int64")
    return df


def cargar_metricas(raiz):
    ruta = Path(raiz) / "reports"
    return {
        "regresion": pd.read_csv(ruta / "metricas_regresion.csv", index_col=0),
        "clasificacion": pd.read_csv(ruta / "metricas_clasificacion.csv", index_col=0),
        "clustering": pd.read_csv(ruta / "metricas_clustering.csv"),
        "informacion": json.loads((ruta / "informacion_modelos.json").read_text(encoding="utf-8")),
    }


def filtrar_pedidos(df, moneda, fechas, categorias=None, edad=None, cantidad=None, ventas=None):
    """Categorías vacías significan cero resultados; se utiliza una sola moneda."""
    mask = df["currency"].eq(moneda)
    if fechas is not None:
        inicio, fin = pd.Timestamp(fechas[0]), pd.Timestamp(fechas[1])
        mask &= df["order_date"].between(inicio, fin)
    for columna, valores in (categorias or {}).items():
        mask &= df[columna].isin(valores)
    for columna, rango in [("customer_age", edad), ("quantity", cantidad), ("net_sales", ventas)]:
        if rango is not None:
            mask &= df[columna].between(rango[0], rango[1])
    return df.loc[mask].copy()


def calcular_kpis(df):
    monedas = df["currency"].dropna().unique()
    if len(monedas) > 1:
        raise ValueError("Los importes deben calcularse para una sola moneda.")
    finalizados = df["order_status"].isin(["Completed", "Returned"])
    completados = df.loc[df["order_status"].eq("Completed")]
    tasa = float(df.loc[finalizados, "order_status"].eq("Returned").mean()) if finalizados.any() else None
    return {
        "pedidos": int(len(df)),
        "clientes": int(df["customer_id"].nunique()),
        "ventas_completadas": float(completados["net_sales"].sum()),
        "ticket_completado": float(completados["net_sales"].mean()) if len(completados) else None,
        "tasa_devolucion": tasa,
        "pedidos_finalizados": int(finalizados.sum()),
    }


def perfiles_clusters(df):
    datos = df.loc[df["currency"].eq("USD") & df["cluster_kmeans"].notna()]
    return (datos.groupby(["cluster_kmeans", "perfil_comercial"], observed=True)
            .agg(pedidos=("order_id", "size"), edad_promedio=("customer_age", "mean"),
                 unidades_promedio=("quantity", "mean"), venta_promedio_usd=("net_sales", "mean"))
            .reset_index())
'''
ruta.parent.mkdir(parents=True, exist_ok=True)
ruta.write_text(contenido, encoding="utf-8")
print("Creado:", ruta)

Creado: /workspace/scratch/95ce581e477a/proyecto_fase4/src/datos.py


## 5. Crear la inferencia con los modelos existentes

El formulario necesita los diez predictores de regresión y la moneda adicional de
clasificación. El código deriva mes y fin de semana a partir de la fecha. Reutiliza
el `Pipeline` completo y el umbral 0,491138 elegido en validación.

La segmentación necesita la venta ya observada: no asignamos un perfil nuevo usando
como si fuera real el importe estimado por regresión. El mapa conserva el escalador,
K-Means y PCA de Fase 3; no se vuelven a ajustar al aplicar filtros.

In [6]:
ruta = PROYECTO / "src/prediccion.py"
contenido = r'''"""Inferencia con los pipelines y transformadores exportados en la Fase 3."""
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import sklearn


def cargar_modelos(raiz):
    if sklearn.__version__ != "1.6.1":
        raise RuntimeError("Instala scikit-learn==1.6.1 antes de cargar los modelos de Fase 3.")
    ruta = Path(raiz) / "models"
    # Estos archivos proceden del ZIP del proyecto y son de confianza.
    return {
        "regresion": joblib.load(ruta / "modelo_regresion.joblib"),
        "clasificacion": joblib.load(ruta / "modelo_clasificacion.joblib"),
        "clustering": joblib.load(ruta / "modelo_clustering.joblib"),
    }


def construir_entrada(valores, fecha):
    """Deriva mes y fin de semana con la misma definición de la Fase 2."""
    fila = dict(valores)
    fecha = pd.Timestamp(fecha)
    fila["order_month"] = int(fecha.month)
    fila["is_weekend"] = int(fecha.dayofweek >= 5)
    if not 18 <= int(fila["customer_age"]) <= 74:
        raise ValueError("La edad debe estar entre 18 y 74 años, el rango observado.")
    if not 1 <= int(fila["quantity"]) <= 24:
        raise ValueError("La cantidad debe estar entre 1 y 24, el rango observado.")
    return pd.DataFrame([fila])


def predecir_pedido(modelos, entrada):
    clf = modelos["clasificacion"]
    pipeline = clf["pipeline"]
    indice_positivo = int(np.flatnonzero(np.asarray(pipeline.classes_) == 1)[0])
    score = float(pipeline.predict_proba(entrada[clf["variables"]])[0, indice_positivo])
    umbral = float(clf["umbral"])
    resultado = {
        "puntuacion_devolucion": score,
        "umbral": umbral,
        "alerta_exploratoria": bool(score >= umbral),
        "venta_estimada_usd": None,
    }
    if entrada.iloc[0]["currency"] == "USD":
        reg = modelos["regresion"]
        resultado["venta_estimada_usd"] = float(reg["pipeline"].predict(entrada[reg["variables"]])[0])
    return resultado


def proyectar_clusters(modelos, df):
    """Transforma y proyecta sin fit: conserva la geometría de la Fase 3."""
    cluster = modelos["clustering"]
    if "currency" in df and not df["currency"].eq("USD").all():
        raise ValueError("La segmentación está definida exclusivamente para USD.")
    if df["quantity"].le(0).any() or df["net_sales"].lt(0).any():
        raise ValueError("La segmentación necesita cantidades positivas y ventas no negativas.")
    x = pd.DataFrame({
        "customer_age": df["customer_age"],
        "log_quantity": np.log1p(df["quantity"]),
        "log_net_sales": np.log1p(df["net_sales"]),
    }, index=df.index)
    x_std = cluster["escalador"].transform(x[cluster["variables"]])
    etiquetas = cluster["modelo"].predict(x_std)
    pca = cluster["pca"].transform(x_std)
    salida = df.copy()
    salida["cluster_kmeans"] = etiquetas
    salida["perfil_comercial"] = pd.Series(etiquetas, index=df.index).map(cluster["nombres_clusters"])
    salida["PCA 1"] = pca[:, 0]
    salida["PCA 2"] = pca[:, 1]
    return salida
'''
ruta.parent.mkdir(parents=True, exist_ok=True)
ruta.write_text(contenido, encoding="utf-8")
print("Creado:", ruta)

Creado: /workspace/scratch/95ce581e477a/proyecto_fase4/src/prediccion.py


## 6. Probar una predicción antes de construir la interfaz

Tomamos un pedido en USD como ejemplo. Puedes cambiar sus características para
probar otro escenario. Esta demostración no evalúa rendimiento; la evaluación válida
se reproduce con el conjunto de prueba en el paso 9.

El MAE de prueba es una referencia del error medio, no un intervalo de confianza.
La puntuación de devolución tampoco se interpreta como probabilidad calibrada.

In [7]:
sys.path.insert(0, str(PROYECTO.resolve()))
from src.datos import cargar_datos, cargar_metricas, calcular_kpis, perfiles_clusters
from src.prediccion import cargar_modelos, construir_entrada, predecir_pedido, proyectar_clusters

df = cargar_datos(PROYECTO)
modelos = cargar_modelos(PROYECTO)
metricas = cargar_metricas(PROYECTO)
ejemplo = df.loc[df.currency.eq("USD")].iloc[0]
entrada = construir_entrada(ejemplo.to_dict(), ejemplo.order_date)
resultado = predecir_pedido(modelos, entrada)
display(entrada)
display(pd.Series(resultado, name="Predicción del ejemplo"))
display(pd.Series(calcular_kpis(df.loc[df.currency.eq("USD")]), name="KPI de pedidos USD"))
display(perfiles_clusters(df))

,indice_origen,order_id,customer_id,order_date,currency,customer_age,customer_segment,sales_channel,quantity,net_sales,order_status,customer_country,payment_method,shipping_method,marketing_channel,estimated_delivery_days,delivery_days,loyalty_points_redeemed,review_sentiment,loyalty_points_log,order_month,is_weekend,age_group,cluster_kmeans,perfil_comercial
0,0,ORD-301242,CUST-003102,2023-11-06,USD,55,Consumer,Mobile App,5,945.4,Completed,USA,Digital Wallet,Standard,Direct,3.0,3.0,44,Positive,3.806662,11,0,45-59,2,Mayores - alto valor


puntuacion_devolucion      0.558396
umbral                     0.491138
alerta_exploratoria            True
venta_estimada_usd       992.647504
Name: Predicción del ejemplo, dtype: object

pedidos                8.260000e+04
clientes               1.487900e+04
ventas_completadas     8.334413e+07
ticket_completado      1.226046e+03
tasa_devolucion        7.617249e-02
pedidos_finalizados    7.358300e+04
Name: KPI de pedidos USD, dtype: float64

,cluster_kmeans,perfil_comercial,pedidos,edad_promedio,unidades_promedio,venta_promedio_usd
0,0,Jóvenes - alto valor,27470,31.099272,6.959410,1602.696009
1,1,Mayores - menor valor,18202,60.311504,3.136578,531.158051
2,2,Mayores - alto valor,22639,59.561288,8.092760,1957.616183
3,3,Jóvenes - menor valor,14289,33.959969,2.224578,295.151541


## 7. Construir el dashboard interactivo

La siguiente celda escribe `dashboard/app.py`, el archivo que Streamlit ejecutará.
No esperes que `st.title()` y los controles se rendericen como una aplicación dentro
de una salida normal de Colab: la interfaz se abre desde un servidor Streamlit.

| Vista | Contenido |
|---|---|
| Resumen comercial | Filtros, cinco KPI, tendencia mensual, estados y descarga |
| Simular un pedido | Predicción de venta USD y puntuación de devolución |
| Segmentos de pedidos | Mapa PCA interactivo y promedios de los grupos |
| Resultados y alcance | Métricas de prueba, comparación de clustering y límites |

Los filtros del panel lateral afectan al resumen y los segmentos. El formulario
utiliza sus propias entradas. Las métricas del modelo permanecen fijas.

In [8]:
ruta = PROYECTO / "dashboard/app.py"
contenido = r'''"""Ejecutar desde la raíz: python -m streamlit run dashboard/app.py"""
from pathlib import Path
from datetime import date
import sys
import pandas as pd
import plotly.express as px
import streamlit as st

RAIZ = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(RAIZ))
from src.datos import cargar_datos, cargar_metricas, filtrar_pedidos, calcular_kpis, perfiles_clusters
from src.prediccion import cargar_modelos, construir_entrada, predecir_pedido, proyectar_clusters

st.set_page_config(page_title="Pedidos | Ciencia de Datos", page_icon="📊", layout="wide")
st.markdown("""<style>
.stApp {background: #f8fafc;}
h1, h2, h3 {color: #13243a;}
[data-testid="stMetric"] {background: white; padding: 16px; border-radius: 12px;
 border: 1px solid #e2e8f0;}
</style>""", unsafe_allow_html=True)


@st.cache_data(show_spinner=False)
def datos_dashboard():
    return cargar_datos(RAIZ)


@st.cache_data(show_spinner=False)
def metricas_dashboard():
    return cargar_metricas(RAIZ)


@st.cache_resource(show_spinner=False)
def modelos_dashboard():
    return cargar_modelos(RAIZ)


df = datos_dashboard()
metricas = metricas_dashboard()
st.title("Pedidos, ventas y perfiles de compra")
st.caption("Proyecto integrador · Jennifer Juliana Bayona Rojas · Universidad de La Salle")
st.info("Datos simulados de comercio electrónico. Los resultados describen este conjunto de datos.")

with st.sidebar:
    st.header("Explorar pedidos")
    st.caption("Estos filtros se aplican al resumen y los segmentos. El simulador tiene sus propias entradas.")
    monedas = ["USD"] + sorted(set(df.currency) - {"USD"})
    moneda = st.selectbox("Moneda de los importes", monedas, key="f_moneda")
    base_moneda = df.loc[df.currency.eq(moneda)]
    fecha_min, fecha_max = df.order_date.min().date(), df.order_date.max().date()
    fechas = st.date_input("Periodo de los pedidos", (fecha_min, fecha_max),
                           min_value=fecha_min, max_value=fecha_max, key="f_fechas")
    categorias = {}
    for columna, etiqueta in [("sales_channel", "Canal de venta"), ("customer_country", "País"),
                              ("customer_segment", "Segmento del cliente"), ("order_status", "Estado del pedido")]:
        opciones = sorted(df[columna].dropna().unique().tolist())
        categorias[columna] = st.multiselect(etiqueta, opciones, default=opciones, key="f_" + columna)
    edad = st.slider("Edad del cliente", 18, 74, (18, 74), key="f_edad")
    cantidad = st.slider("Unidades por pedido", 1, 24, (1, 24), key="f_cantidad")
    limite_venta = float(base_moneda.net_sales.max())
    ventas = st.slider(f"Importe registrado ({moneda})", 0.0, limite_venta, (0.0, limite_venta), key="f_ventas_" + moneda)

if len(fechas) != 2:
    st.warning("Completa las dos fechas del periodo para actualizar el resumen.")
    filtrado = df.iloc[:0].copy()
else:
    filtrado = filtrar_pedidos(df, moneda, fechas, categorias, edad, cantidad, ventas)

resumen_tab, pred_tab, cluster_tab, metodo_tab = st.tabs([
    "Resumen comercial", "Simular un pedido", "Segmentos de pedidos", "Resultados y alcance"
])

with resumen_tab:
    st.subheader(f"Resumen de pedidos en {moneda}")
    if filtrado.empty:
        st.info("No hay pedidos para esta combinación de filtros. Amplía el periodo o las categorías.")
    else:
        kpis = calcular_kpis(filtrado)
        cols = st.columns(5)
        cols[0].metric("Pedidos", f"{kpis['pedidos']:,}")
        cols[1].metric("Clientes distintos", f"{kpis['clientes']:,}")
        cols[2].metric(f"Venta de completados ({moneda})", f"{kpis['ventas_completadas']:,.2f}")
        ticket = kpis["ticket_completado"]
        cols[3].metric(f"Ticket de completados ({moneda})", "Sin datos" if ticket is None else f"{ticket:,.2f}")
        tasa = kpis["tasa_devolucion"]
        cols[4].metric("Devoluciones / finalizados", "Sin datos" if tasa is None else f"{tasa:.2%}")
        st.caption("Venta y ticket: solo pedidos Completed. Tasa: Returned / (Completed + Returned) dentro de los filtros. "
                   "Si filtras por estado, cambia también ese denominador. Los importes no representan beneficio ni flujo de caja.")
        completados = filtrado.loc[filtrado.order_status.eq("Completed")].copy()
        izq, der = st.columns([2, 1])
        if not completados.empty:
            completados["Mes"] = completados.order_date.dt.to_period("M").dt.to_timestamp()
            mensual = completados.groupby("Mes", as_index=False).net_sales.sum()
            izq.plotly_chart(px.line(mensual, x="Mes", y="net_sales", markers=True,
                                    title=f"Venta mensual de pedidos completados ({moneda})",
                                    labels={"net_sales": f"Venta ({moneda})"}), width="stretch")
        estados = filtrado.order_status.value_counts().rename_axis("Estado").reset_index(name="Pedidos")
        der.plotly_chart(px.bar(estados, x="Estado", y="Pedidos", title="Estados de los pedidos",
                               color="Estado"), width="stretch")
        st.markdown("**Preguntas que orientan la lectura**")
        st.write("¿Cómo varían los importes y el ticket por canal y periodo? ¿Qué cambia en la proporción "
                 "de devoluciones? Estas diferencias describen asociaciones; no prueban efectos causales.")
        vista = filtrado[["order_id", "order_date", "currency", "sales_channel", "customer_country",
                          "quantity", "net_sales", "order_status", "perfil_comercial"]]
        st.dataframe(vista.head(500), hide_index=True, width="stretch")
        st.caption("La tabla muestra como máximo 500 pedidos; la descarga contiene todos los pedidos filtrados.")
        st.download_button("Descargar pedidos filtrados", vista.to_csv(index=False).encode("utf-8-sig"),
                           file_name=f"pedidos_{moneda}.csv", mime="text/csv")

with pred_tab:
    st.subheader("Simular las características de un pedido")
    st.write("Completa los datos disponibles al registrar la compra. El mes y el indicador de fin de semana "
             "se calculan a partir de la fecha.")
    with st.form("form_prediccion"):
        a, b, c = st.columns(3)
        fecha_pred = a.date_input("Fecha del pedido", date(2025, 6, 15), key="p_fecha")
        edad_pred = b.number_input("Edad", 18, 74, 35, step=1, key="p_edad")
        cantidad_pred = c.number_input("Cantidad de unidades", 1, 24, 5, step=1, key="p_cantidad")
        valores = {"customer_age": int(edad_pred), "quantity": int(cantidad_pred)}
        campos = [("currency", "Moneda"), ("customer_segment", "Segmento del cliente"),
                  ("sales_channel", "Canal de venta"), ("customer_country", "País del cliente"),
                  ("payment_method", "Método de pago"), ("shipping_method", "Tipo de envío"),
                  ("marketing_channel", "Canal de marketing")]
        for i, (campo, etiqueta) in enumerate(campos):
            opciones = sorted(df[campo].dropna().unique().tolist())
            indice = opciones.index("USD") if campo == "currency" else 0
            valores[campo] = [a, b, c][i % 3].selectbox(etiqueta, opciones, index=indice, key="p_" + campo)
        enviado = st.form_submit_button("Obtener predicción", type="primary")
    if enviado:
        entrada = construir_entrada(valores, fecha_pred)
        resultado = predecir_pedido(modelos_dashboard(), entrada)
        if resultado["venta_estimada_usd"] is not None:
            st.metric("Venta neta estimada (USD)", f"{resultado['venta_estimada_usd']:,.2f}")
            mae = metricas["regresion"].loc["Prueba", "MAE"]
            st.caption(f"Error absoluto medio en prueba: {mae:,.2f} USD. Es un promedio histórico del error; "
                       "no es un intervalo de confianza para esta predicción.")
        else:
            st.info("El modelo de ventas se entrenó únicamente en USD. Para esta moneda se muestra la clasificación exploratoria.")
        st.metric("Puntuación del clasificador de devolución", f"{resultado['puntuacion_devolucion']:.4f}")
        etiqueta = "Alerta exploratoria" if resultado["alerta_exploratoria"] else "Sin alerta exploratoria"
        st.write(f"**{etiqueta}** · Umbral de Fase 3: {resultado['umbral']:.4f}")
        st.warning("La puntuación no es una probabilidad calibrada. En prueba, la precisión fue 8,4 % "
                   "y el AUC-ROC fue 0,561. Este clasificador tiene capacidad limitada y genera muchas falsas alertas.")
        if not fecha_min <= fecha_pred <= fecha_max:
            st.info("La fecha está fuera del periodo observado (2021–2025). La predicción temporal requiere validación adicional.")
        with st.expander("Ver entradas utilizadas"):
            st.dataframe(entrada, hide_index=True)

with cluster_tab:
    st.subheader("Cuatro perfiles de pedidos en USD")
    st.write("Los grupos describen pedidos, no clientes permanentes. Se asignan después de conocer la venta neta.")
    st.caption("Se usan edad, log(1 + unidades) y log(1 + venta neta), junto con el escalador, K-Means y PCA de Fase 3.")
    if moneda != "USD":
        st.info("Selecciona USD en los filtros para explorar los segmentos definidos en el proyecto.")
    elif filtrado.empty:
        st.info("No hay pedidos en USD para esta combinación de filtros.")
    else:
        nombres = sorted(filtrado.perfil_comercial.dropna().unique().tolist())
        seleccion = st.multiselect("Perfiles que quieres visualizar", nombres, default=nombres, key="c_perfiles")
        grupos = filtrado.loc[filtrado.perfil_comercial.isin(seleccion)].copy()
        if grupos.empty:
            st.info("Selecciona al menos un perfil para mostrar el mapa.")
        else:
            perfiles = perfiles_clusters(grupos)
            st.dataframe(perfiles.round(2), hide_index=True, width="stretch")
            muestra = grupos.sample(n=min(5000, len(grupos)), random_state=42)
            proyectados = proyectar_clusters(modelos_dashboard(), muestra)
            fig = px.scatter(proyectados, x="PCA 1", y="PCA 2", color="perfil_comercial",
                             hover_data=["customer_age", "quantity", "net_sales", "order_date"],
                             title="Mapa interactivo de perfiles de pedidos", opacity=0.55,
                             labels={"perfil_comercial": "Perfil", "customer_age": "Edad",
                                     "quantity": "Unidades", "net_sales": "Venta (USD)"})
            st.plotly_chart(fig, width="stretch")
            st.caption(f"Mapa: muestra reproducible de {len(muestra):,} de {len(grupos):,} pedidos seleccionados. "
                       "La tabla de perfiles usa todos los pedidos seleccionados. PCA conserva el 89,85 % de la variabilidad.")
            acciones = pd.DataFrame([
                ["Jóvenes - alto valor", "Evaluar paquetes de productos complementarios"],
                ["Mayores - menor valor", "Evaluar un producto complementario a precio regular"],
                ["Mayores - alto valor", "Evaluar un beneficio para la compra siguiente"],
                ["Jóvenes - menor valor", "Evaluar un combo de entrada de bajo precio"],
            ], columns=["Perfil del pedido", "Hipótesis para un piloto"])
            st.dataframe(acciones, hide_index=True, width="stretch")
            st.caption("Las acciones requieren un piloto con datos reales. No se ha medido un aumento de ventas.")

with metodo_tab:
    st.subheader("Rendimiento del modelo y alcance de la solución")
    st.caption("Métricas guardadas en Fase 3. Son resultados de las particiones originales y no cambian con los filtros.")
    r = metricas["regresion"].loc["Prueba"]
    c = metricas["clasificacion"].loc["Prueba"]
    col = st.columns(4)
    col[0].metric("RMSE de regresión (USD)", f"{r['RMSE']:,.2f}")
    col[1].metric("R² de regresión", f"{r['R2']:.3f}")
    col[2].metric("AUC-ROC de clasificación", f"{c['AUC-ROC']:.3f}")
    col[3].metric("F1 de clasificación", f"{c['F1-Score']:.3f}")
    st.markdown("**Regresión: Random Forest optimizado**")
    st.dataframe(metricas["regresion"].round(4), width="stretch")
    st.markdown("**Clasificación exploratoria: Random Forest y umbral fijado en validación**")
    st.dataframe(metricas["clasificacion"].round(4), width="stretch")
    st.markdown("**Comparación de segmentaciones**")
    st.dataframe(metricas["clustering"].round(4), hide_index=True, width="stretch")
    st.caption("K-Means se ajustó con 82.600 pedidos en USD y DBSCAN con una muestra de 8.000. "
               "Las siluetas exportadas se comparan sobre las mismas 7.932 filas sin ruido de DBSCAN.")
    st.write("La partición supervisada original es 70 % entrenamiento, 15 % validación y 15 % prueba; "
             "la clasificación es estratificada. La separación se hizo por pedido: hay clientes "
             "compartidos entre particiones y falta una evaluación temporal y por cliente.")
    st.write("La aplicación reutiliza los pipelines exportados. No reajusta modelos al cambiar filtros. "
             "Se excluyeron variables que revelaban el resultado, como entrega, reseña y puntos de fidelización.")
    st.link_button("Consultar el dataset de origen", "https://www.kaggle.com/datasets/datascikhan/e-commerce-sales-and-customer-analytics")
'''
ruta.parent.mkdir(parents=True, exist_ok=True)
ruta.write_text(contenido, encoding="utf-8")
print("Creado:", ruta)

Creado: /workspace/scratch/95ce581e477a/proyecto_fase4/dashboard/app.py


## 8. Preparar las dependencias y la configuración

`requirements.txt` queda en la raíz del repositorio. Las versiones se fijan con
`==` y corresponden al entorno de inferencia comprobado. El archivo original de
dependencias de Fase 3 se conserva por separado en `reports/`.

La versión de Python del despliegue se selecciona en **Advanced settings** de
Streamlit Community Cloud: elige **3.12**. El archivo de configuración define el tema.

In [9]:
ruta = PROYECTO / "requirements.txt"
contenido = r'''# Dashboard e inferencia, verificados con Python 3.12
streamlit==1.51.0
plotly==6.5.0
pandas==2.2.3
numpy==2.3.5
scipy==1.16.3
scikit-learn==1.6.1
joblib==1.5.3
'''
ruta.parent.mkdir(parents=True, exist_ok=True)
ruta.write_text(contenido, encoding="utf-8")
print("Creado:", ruta)

Creado: /workspace/scratch/95ce581e477a/proyecto_fase4/requirements.txt


In [10]:
ruta = PROYECTO / "requirements_notebook.txt"
contenido = r'''-r requirements.txt
nbformat==5.10.4
nbclient==0.10.2
ipykernel==6.30.1
'''
ruta.parent.mkdir(parents=True, exist_ok=True)
ruta.write_text(contenido, encoding="utf-8")
print("Creado:", ruta)

Creado: /workspace/scratch/95ce581e477a/proyecto_fase4/requirements_notebook.txt


In [11]:
ruta = PROYECTO / ".streamlit/config.toml"
contenido = r'''[theme]
primaryColor = "#2563eb"
backgroundColor = "#f8fafc"
secondaryBackgroundColor = "#ffffff"
textColor = "#13243a"
font = "sans serif"

[browser]
gatherUsageStats = false
'''
ruta.parent.mkdir(parents=True, exist_ok=True)
ruta.write_text(contenido, encoding="utf-8")
print("Creado:", ruta)

Creado: /workspace/scratch/95ce581e477a/proyecto_fase4/.streamlit/config.toml


In [12]:
ruta = PROYECTO / ".gitignore"
contenido = r'''__pycache__/
*.pyc
.ipynb_checkpoints/
.venv/
.env
.streamlit/secrets.toml
*.zip
'''
ruta.parent.mkdir(parents=True, exist_ok=True)
ruta.write_text(contenido, encoding="utf-8")
print("Creado:", ruta)

Creado: /workspace/scratch/95ce581e477a/proyecto_fase4/.gitignore


## 9. Verificar integridad y reproducir las métricas

Esta comprobación utiliza los modelos guardados y reconstruye exactamente los splits
de Fase 2. No entrena modelos, no busca hiperparámetros y no cambia el umbral.

Se comprueban las 82.600 etiquetas K-Means, los casos de filtros vacíos, la separación
monetaria y las métricas originales de prueba. Si un `assert` falla, revisa los archivos
y las versiones antes de publicar.

In [13]:
ruta = PROYECTO / "src/verificar.py"
contenido = r'''"""Comprueba integridad, inferencia y reproducción de métricas de prueba."""
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score)

RAIZ = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(RAIZ))
from src.datos import cargar_datos, cargar_metricas, calcular_kpis, filtrar_pedidos
from src.prediccion import cargar_modelos, proyectar_clusters, construir_entrada, predecir_pedido


def verificar(raiz=RAIZ):
    raiz = Path(raiz)
    df = cargar_datos(raiz)
    modelos = cargar_modelos(raiz)
    metricas = cargar_metricas(raiz)
    usd = df.loc[df.currency.eq("USD")].copy()
    supervisados = df.loc[df.order_status.isin(["Completed", "Returned"])].copy()
    assert len(df) == 138116 and len(usd) == 82600
    assert len(supervisados) == 123021
    proyeccion = proyectar_clusters(modelos, usd)
    assert np.array_equal(proyeccion.cluster_kmeans, usd.cluster_kmeans.astype(int))
    assert (proyeccion.perfil_comercial == usd.perfil_comercial).all()
    assert np.isfinite(proyeccion[["PCA 1", "PCA 2"]].to_numpy()).all()
    assert filtrar_pedidos(df, "USD", None, {"sales_channel": []}).empty
    assert calcular_kpis(df.loc[df.order_status.eq("Pending") & df.currency.eq("USD")])["tasa_devolucion"] is None
    try:
        calcular_kpis(df)
    except ValueError:
        pass
    else:
        raise AssertionError("No se deben sumar monedas distintas.")

    # Mismos splits de Fase 2. Solo se predice: no se ajustan ni seleccionan modelos.
    _, temp_reg = train_test_split(usd, test_size=0.30, random_state=42)
    _, test_reg = train_test_split(temp_reg, test_size=0.50, random_state=42)
    reg = modelos["regresion"]
    pred_reg = reg["pipeline"].predict(test_reg[reg["variables"]])
    calculadas_reg = {
        "MAE": mean_absolute_error(test_reg.net_sales, pred_reg),
        "RMSE": float(np.sqrt(mean_squared_error(test_reg.net_sales, pred_reg))),
        "R2": r2_score(test_reg.net_sales, pred_reg),
    }
    for nombre, valor in calculadas_reg.items():
        assert np.isclose(valor, metricas["regresion"].loc["Prueba", nombre], rtol=1e-8, atol=1e-8), (nombre, valor)

    y = supervisados.order_status.eq("Returned").astype(int)
    _, temp_clf = train_test_split(supervisados, test_size=0.30, stratify=y, random_state=42)
    _, test_clf = train_test_split(temp_clf, test_size=0.50,
                                 stratify=temp_clf.order_status.eq("Returned").astype(int), random_state=42)
    clf = modelos["clasificacion"]
    clases = np.asarray(clf["pipeline"].classes_)
    indice_positivo = int(np.flatnonzero(clases == 1)[0])
    scores = clf["pipeline"].predict_proba(test_clf[clf["variables"]])[:, indice_positivo]
    pred_clf = scores >= clf["umbral"]
    real_clf = test_clf.order_status.eq("Returned").astype(int)
    calculadas_clf = {
        "Accuracy": accuracy_score(real_clf, pred_clf),
        "Precision": precision_score(real_clf, pred_clf, zero_division=0),
        "Recall": recall_score(real_clf, pred_clf, zero_division=0),
        "F1-Score": f1_score(real_clf, pred_clf, zero_division=0),
        "AUC-ROC": roc_auc_score(real_clf, scores),
        "Average Precision": average_precision_score(real_clf, scores),
    }
    for nombre, valor in calculadas_clf.items():
        assert np.isclose(valor, metricas["clasificacion"].loc["Prueba", nombre], rtol=1e-8, atol=1e-8), (nombre, valor)

    entrada = construir_entrada(usd.iloc[0].to_dict(), usd.iloc[0].order_date)
    demo = predecir_pedido(modelos, entrada)
    assert demo["venta_estimada_usd"] is not None
    assert 0 <= demo["puntuacion_devolucion"] <= 1
    entrada["currency"] = "EUR"
    assert predecir_pedido(modelos, entrada)["venta_estimada_usd"] is None
    assert np.isclose(modelos["clasificacion"]["umbral"], metricas["informacion"]["umbral_clasificacion"])

    resultado = {
        "integridad_datos": "correcta", "registros": int(len(df)), "pedidos_usd": int(len(usd)),
        "clusters_coincidentes": int(len(proyeccion)), "metricas_prueba_reproducidas": True,
        "regresion_prueba": calculadas_reg, "clasificacion_prueba": calculadas_clf,
        "filas_prueba_regresion": int(len(test_reg)), "filas_prueba_clasificacion": int(len(test_clf)),
        "umbral_clasificacion": float(clf["umbral"]),
        "publicacion_dashboard": "pendiente de publicar y verificar URL",
    }
    (raiz / "reports/verificacion.json").write_text(json.dumps(resultado, ensure_ascii=False, indent=2), encoding="utf-8")
    return resultado


if __name__ == "__main__":
    print(json.dumps(verificar(), ensure_ascii=False, indent=2))
'''
ruta.parent.mkdir(parents=True, exist_ok=True)
ruta.write_text(contenido, encoding="utf-8")
print("Creado:", ruta)

Creado: /workspace/scratch/95ce581e477a/proyecto_fase4/src/verificar.py


In [14]:
verificacion = subprocess.run(
    [sys.executable, str(PROYECTO / "src/verificar.py")],
    cwd=PROYECTO, text=True, capture_output=True
)
if verificacion.returncode:
    print(verificacion.stderr)
    raise RuntimeError("La verificación falló. Corrige el error antes de publicar.")
print(verificacion.stdout)

{
  "integridad_datos": "correcta",
  "registros": 138116,
  "pedidos_usd": 82600,
  "clusters_coincidentes": 82600,
  "metricas_prueba_reproducidas": true,
  "regresion_prueba": {
    "MAE": 608.7491969663101,
    "RMSE": 885.1159298841429,
    "R2": 0.3824172124574664
  },
  "clasificacion_prueba": {
    "Accuracy": 0.33000975398287635,
    "Precision": 0.0838150289017341,
    "Recall": 0.776056338028169,
    "F1-Score": 0.1512904997254256,
    "AUC-ROC": 0.560520218882864,
    "Average Precision": 0.0999195764409961
  },
  "filas_prueba_regresion": 12390,
  "filas_prueba_clasificacion": 18454,
  "umbral_clasificacion": 0.4911380775074184,
  "publicacion_dashboard": "pendiente de publicar y verificar URL"
}



### 9.1 Prueba de la interfaz

`AppTest` simula la ejecución de Streamlit para comprobar que la aplicación abre,
responde a filtros y produce una predicción. No reemplaza la prueba desde la URL
pública, que se realizará después del despliegue.

In [15]:
from streamlit.testing.v1 import AppTest

app = AppTest.from_file(str(PROYECTO / "dashboard/app.py"), default_timeout=60).run()
assert not app.exception, [e.value for e in app.exception]
assert app.metric[0].value == "82,600"
app.multiselect(key="f_sales_channel").set_value([]).run()
assert not app.exception
assert any("No hay pedidos" in elemento.value for elemento in app.info)
app.multiselect(key="f_sales_channel").set_value(sorted(df.sales_channel.unique().tolist())).run()
app.button(key="FormSubmitter:form_prediccion-Obtener predicción").click().run()
assert not app.exception
assert any(m.label == "Venta neta estimada (USD)" for m in app.metric)
app.selectbox(key="f_moneda").set_value("EUR").run()
assert not app.exception
assert any("Selecciona USD" in elemento.value for elemento in app.info)
print("Aplicación comprobada: apertura, filtros vacíos, predicción USD y alcance de clústeres.")

Aplicación comprobada: apertura, filtros vacíos, predicción USD y alcance de clústeres.


2026-11-25 17:48:31.351 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


## 10. Redactar la narrativa, el informe y la sustentación

La historia del dashboard conecta tres preguntas: cuánto vende un pedido, qué señal
existe sobre su devolución y qué perfiles de compra aparecen. Los resultados deben
presentarse con sus límites y con decisiones comerciales planteadas como hipótesis.

La siguiente documentación está basada en tus resultados reales. El informe se deja
como borrador para incorporar URL pública, evidencia de funcionamiento y revisión
final antes de exportarlo a PDF. El guion distribuye los 12 minutos exigidos.

In [16]:
ruta = PROYECTO / "README.md"
contenido = r'''# Dashboard de pedidos de comercio electrónico

Proyecto integrador de Ciencia de Datos e Inteligencia Artificial.
Jennifer Juliana Bayona Rojas · Universidad de La Salle · Fase 4.

## Qué contiene

Dashboard en Streamlit con filtros por fecha, moneda, país, canal, segmento, estado,
edad, unidades e importe. Incluye indicadores, predicción con los modelos de Fase 3,
mapa interactivo PCA de K-Means y resultados técnicos.

Los datos son transacciones simuladas. La regresión y el clustering están definidos
para USD. La clasificación es exploratoria: AUC-ROC de prueba 0,5605 y precisión 8,38 %.
La puntuación del clasificador no se presenta como una probabilidad calibrada.

## Empezar por el notebook

Abre `notebooks/Fase_4_PROYECTO.ipynb` en Jupyter o Google Colab y sigue las celdas en
orden. El notebook explica cada paso y contiene el código que genera esta aplicación.
Puedes usar esta carpeta ya preparada o trabajar desde `artefactos_fase3.zip`.

## Ejecutar el dashboard en tu computador

Usa Python 3.12. Desde esta carpeta:

```bash
python -m venv .venv
```

Activa el entorno. En Windows: `.venv\Scripts\activate`. En macOS/Linux:
`source .venv/bin/activate`. Después:

```bash
python -m pip install -r requirements.txt
python src/verificar.py
python -m streamlit run dashboard/app.py
```

Abre la dirección local que indica Streamlit. Para editar/ejecutar notebooks también
puedes instalar `python -m pip install -r requirements_notebook.txt`.

## Publicar con GitHub y Streamlit Community Cloud

1. Crea en https://github.com/new un repositorio público llamado, por ejemplo,
   `proyecto-ciencia-datos-ecommerce`. La guía de la asignatura pide que sea público.
2. Descomprime la entrega y sube **los archivos descomprimidos** conservando las
   carpetas. `requirements.txt` debe quedar en la raíz y el script en `dashboard/app.py`.
   Usa **Add file → Upload files** o Git. Subir solamente el ZIP no permite ejecutar la app.
3. Comprueba que se ven `data/`, `models/`, `src/`, `dashboard/`, `notebooks/` y `reports/`.
4. Entra en https://share.streamlit.io/ y conecta la cuenta de GitHub que tiene el repo.
5. Selecciona **Create app** y la opción para una app existente. Indica:

   | Campo | Valor |
   |---|---|
   | Repository | tu-usuario/proyecto-ciencia-datos-ecommerce |
   | Branch | main (o la rama en la que subiste los archivos) |
   | Main file path | dashboard/app.py |
   | Advanced settings → Python version | 3.12 |

6. Pulsa **Deploy**. La instalación puede tardar unos minutos. Revisa los logs si falla.
7. Abre la URL real que te entregue Streamlit. Prueba filtros, una predicción en USD,
   una clasificación con otra moneda y el mapa de clústeres.
8. Comprueba el acceso en una ventana privada. Si solicita iniciar sesión, revisa la
   configuración de acceso para que el docente pueda abrirlo como esperas.
9. Añade la URL real al README, al notebook y al informe; registra también la URL del
   repositorio y el commit presentado. No se ha publicado una URL desde esta entrega.

GitHub almacena el código y Streamlit Community Cloud ejecuta Python. GitHub Pages
publica sitios estáticos y no ejecuta este servidor Streamlit.

Para actualizar la app, sube los cambios al mismo repositorio. Conserva las versiones
fijadas para evitar cambios inesperados al cargar los modelos.

## Archivos principales

| Ruta | Propósito |
|---|---|
| data/dataset_procesado.csv | Los 138.116 pedidos de la Fase 3 |
| data/segmentos_kmeans.csv | Segmentos de los 82.600 pedidos en USD |
| data/base_depurada_fase1.csv | Copia de los datos de entrada de Fase 2 |
| models/ | Modelos y transformadores exportados en Fase 3 |
| notebooks/ | Notebooks de las cuatro fases |
| src/datos.py | Unión validada, filtros, indicadores y perfiles |
| src/prediccion.py | Inferencia, umbral original y proyección PCA |
| src/verificar.py | Integridad y reproducción de métricas de prueba |
| dashboard/app.py | Aplicación web |
| reports/ | Métricas, borrador del informe y guion de sustentación |
| requirements.txt | Dependencias fijadas del dashboard y la inferencia |

## Reproducibilidad y alcance

La inferencia se comprobó con Python 3.12, scikit-learn 1.6.1 y las versiones de
`requirements.txt`. Esta es la misma versión de scikit-learn de los modelos guardados.
El script `src/verificar.py` reconstruye las divisiones originales 70/15/15 usando
`random_state=42` y reproduce las métricas de prueba guardadas, sin volver a entrenar.
También comprueba que los 82.600 clústeres coincidan con los exportados.

`reports/requirements_fase3_original.txt` conserva el entorno registrado en el notebook
original, que difiere del entorno probado de inferencia. No se usa para desplegar.
Los modelos de DBSCAN y MLP se conservan como evidencia; el dashboard sirve los
Random Forest seleccionados y K-Means. No necesita instalar PyTorch para predecir.

Los notebooks de Fase 1–3 se conservaron como fueron adjuntados. La repetición de todo
el entrenamiento depende de los archivos de origen y de esos entornos originales.
En especial, el notebook original de Fase 2 descarga `base_datos.csv`; puedes usar la
copia local `data/base_depurada_fase1.csv` en esa celda. El notebook original de Fase 3
espera `fase2_para_fase3.joblib`, que se obtiene al ejecutar Fase 2. El despliegue y la
Fase 4 funcionan directamente con los archivos incluidos y no descargan esos datos.

Las métricas de modelo corresponden a las particiones originales. Los filtros cambian
indicadores descriptivos, no las métricas de prueba. No se ajustan modelos en la app.
La separación por pedido permite clientes compartidos entre particiones; faltan
evaluaciones temporales y por cliente. Los perfiles son de pedidos y no identidades
permanentes de clientes. Las hipótesis comerciales requieren un piloto real.

## Procedencia

Los archivos recibidos para el proyecto se registran con SHA-256 en
`reports/procedencia_archivos.json`. Según la ficha de Fase 1: E-Commerce Sales
Analytics Dataset, Shair Khan (datascikhan), transacciones simuladas y licencia CC0
declarada allí. Fuente: https://www.kaggle.com/datasets/datascikhan/e-commerce-sales-and-customer-analytics
La información de licencia debe revisarse en la fuente al preparar la entrega pública.

## Documentación oficial de publicación

- https://docs.streamlit.io/deploy/streamlit-community-cloud
- https://docs.streamlit.io/deploy/streamlit-community-cloud/deploy-your-app/deploy
- https://docs.streamlit.io/deploy/streamlit-community-cloud/deploy-your-app/app-dependencies
- https://docs.github.com/en/pages/getting-started-with-github-pages/creating-a-github-pages-site

## Pendientes de entrega

- Publicar la aplicación y registrar sus URL reales.
- Ejecutar una demostración desde la URL publicada y guardar la evidencia.
- Revisar y convertir el borrador técnico a PDF con los enlaces finales.
- Descargar el notebook ejecutado de Colab y subir esa versión a `notebooks/`.
- Ensayar la exposición de 12 minutos y las preguntas.
'''
ruta.parent.mkdir(parents=True, exist_ok=True)
ruta.write_text(contenido, encoding="utf-8")
print("Creado:", ruta)

Creado: /workspace/scratch/95ce581e477a/proyecto_fase4/README.md


In [17]:
ruta = PROYECTO / "reports/informe_final_borrador.md"
contenido = r'''# Estimación de ventas, devoluciones y perfiles de pedidos de comercio electrónico

**Autora:** Jennifer Juliana Bayona Rojas.  
**Programa:** Maestría en Inteligencia Artificial, Universidad de La Salle.  
**Estado:** borrador técnico basado en los archivos de Fase 1–3 y la implementación local de Fase 4.
La publicación y la demostración desde una URL pública están pendientes.

## Resumen

Se construyó una solución analítica para estimar la venta neta de pedidos en USD,
explorar la clasificación de devoluciones y describir perfiles de pedidos. El conjunto
procesado contiene 138.116 pedidos; 82.600 están expresados en USD y 123.021 tienen
estado Completed o Returned. Se reutilizaron los pipelines y resultados exportados
en la Fase 3 para desarrollar una aplicación Streamlit con filtros, indicadores,
predicción interactiva y segmentación visual. Random Forest de regresión obtuvo en
prueba MAE de 608,75 USD, RMSE de 885,12 USD y R² de 0,3824. El clasificador de
devoluciones alcanzó AUC-ROC de 0,5605 y precisión de 8,38 %, por lo que su alcance
es exploratorio. K-Means con cuatro grupos se utilizó para visualizar perfiles de
pedidos en USD. Los datos son simulados y no permiten afirmar impacto comercial real.

## 1. Introducción

La información de pedidos permite explorar cómo se relacionan las características
de compra con el importe registrado y las devoluciones. La pregunta del proyecto
es cómo utilizar esas características para estimar la venta neta en USD, identificar
señales asociadas con devoluciones y describir perfiles de pedidos.

El objetivo es integrar preparación, evaluación e interacción en una aplicación
que permita consultar resultados y reconocer sus límites. La unidad principal es
un pedido; un mismo cliente puede aparecer en varias transacciones.

## 2. Metodología

### 2.1 Datos y preparación

Se utilizó la tabla procesada del E-Commerce Sales Analytics Dataset de Kaggle.
Según la ficha de Fase 1, los datos representan transacciones simuladas entre
2021-01-01 y 2025-12-31. La base depurada tiene 18 columnas y 138.116 pedidos.
La tabla exportada de Fase 3 incorpora variables derivadas y `indice_origen`.
La regresión y el clustering se restringen a USD para conservar una unidad monetaria
comparable. No se suman monedas distintas ni se simulan conversiones de divisas.

En Fase 2 se revisó el mecanismo de ausencia de entrega y reseñas, que se relaciona
con el estado del pedido. Esos campos y los puntos de fidelización se excluyeron de
los predictores finales por su disponibilidad y posible fuga de información.
Se crearon mes del pedido, indicador de fin de semana y grupo de edad. La codificación
y el preprocesamiento supervisados se integraron en pipelines ajustados con entrenamiento.

### 2.2 Modelos y evaluación

Se conservaron divisiones de 70 % entrenamiento, 15 % validación y 15 % prueba,
con semilla 42; la clasificación utiliza estratificación. En Fase 3 se compararon
Ridge, Random Forest y una MLP para regresión, y Random Forest y SVM para clasificación.
Los Random Forest seleccionados se optimizaron con validación cruzada; el umbral
clasificador 0,491138 se seleccionó en validación y se mantuvo fijo en prueba.

Para clustering se transformaron cantidad e importe con log1p y se estandarizaron
junto con edad. Se compararon K-Means y DBSCAN. Se eligió K=4 para el mapa principal,
considerando el codo y la interpretación, aunque la silueta favorecía otra configuración
según la narrativa de Fase 3. PCA se utilizó únicamente para representar los grupos.

### 2.3 Implementación de Fase 4

Se implementaron cuatro vistas: resumen comercial, simulador, segmentos y resultados.
Los filtros incluyen fecha, moneda, país, canal, segmento, estado, edad, unidades
e importe. La unión de clústeres se valida mediante `indice_origen` y las variables
originales. Se reutilizan los pipelines, el umbral, el escalador, K-Means y PCA,
sin ajustes durante la interacción.

La venta y el ticket corresponden a pedidos Completed dentro de cada filtro. La tasa
de devolución es Returned / (Completed + Returned) dentro del conjunto seleccionado.
El simulador de ventas se restringe a USD. La salida de clasificación se denomina
puntuación, porque no se ha verificado su calibración. El mapa usa hasta 5.000 puntos
para facilitar la interacción y calcula perfiles con todas las filas seleccionadas.

## 3. Resultados

### 3.1 Regresión en el conjunto de prueba

| Métrica | Resultado |
|---|---:|
| Pedidos de prueba | 12.390 |
| MAE | 608,7492 USD |
| RMSE | 885,1159 USD |
| R² | 0,3824 |

La diferencia de RMSE frente a validación fue de aproximadamente 4,71 USD.
El error medio absoluto aporta una referencia de magnitud; no constituye un
intervalo de confianza para un pedido individual.

### 3.2 Clasificación en el conjunto de prueba

| Métrica | Resultado |
|---|---:|
| Pedidos de prueba | 18.454 |
| Accuracy | 0,3300 |
| Precisión | 0,0838 |
| Sensibilidad | 0,7761 |
| F1 | 0,1513 |
| AUC-ROC | 0,5605 |
| Average Precision | 0,0999 |

Con el umbral seleccionado se detectaron 1.102 de las 1.420 devoluciones reales,
pero 12.046 de las 13.148 alertas fueron falsos positivos, según Fase 3.
El desempeño limita el uso del clasificador como apoyo exploratorio.

### 3.3 Clustering

| Modelo | Configuración | Registros de ajuste | Silueta exportada | Ruido |
|---|---|---:|---:|---:|
| K-Means | K=4 | 82.600 | 0,3028 | 0 % |
| DBSCAN | eps=0,481; min_samples=20 | 8.000 | 0,2081 | 0,85 % |

Las siluetas se compararon sobre las mismas 7.932 observaciones sin ruido de DBSCAN.
Los perfiles de K-Means combinan diferencias de edad, unidades y valor. Las dos
componentes PCA conservan aproximadamente el 89,85 % de la variabilidad transformada.
Los nombres jóvenes/mayores y alto/menor valor son etiquetas descriptivas relativas
a los promedios de los grupos, no límites demográficos o monetarios oficiales.

### 3.4 Validación de la aplicación

El script `src/verificar.py` reproduce las métricas de prueba de los modelos guardados
y contrasta las 82.600 etiquetas K-Means con las exportadas. La versión de inferencia
se fija en `requirements.txt` y usa scikit-learn 1.6.1. La publicación pública se debe
comprobar después del despliegue; una verificación local no prueba acceso remoto.

## 4. Discusión

La regresión captura parte de la variabilidad de ventas, pero conserva errores
considerables para decisiones por pedido. El clasificador tiene una mejora reducida
en discriminación y una elevada tasa de falsas alertas. Un mayor recall no basta
para justificar decisiones operativas sin evaluar sus costos y su precisión.

Los segmentos permiten formular hipótesis para futuras acciones comerciales, como
paquetes complementarios o beneficios para la compra siguiente. No se ha ejecutado
un experimento ni medido incremento de ventas, recompra o rentabilidad. Para evaluar
esas acciones se propone asignar tratamientos por cliente y medir resultados a 30 días.

La partición por pedido permite clientes compartidos entre conjuntos. La evaluación
no mide directamente desempeño con clientes nuevos ni predicción hacia el futuro.
La naturaleza simulada de los datos restringe la generalización a una operación real.

## 5. Conclusiones

Se integraron los resultados del proyecto en una aplicación interactiva que permite
filtrar pedidos, consultar indicadores, simular entradas y visualizar cuatro perfiles
en USD. Los modelos se sirven con sus transformaciones originales y sin reajuste.
Random Forest de regresión ofrece una estimación con precisión moderada; el
clasificador de devoluciones requiere mejoras antes de considerar un uso operativo.
K-Means facilita la lectura de perfiles descriptivos y el planteamiento de pilotos.

## 6. Trabajo futuro

Evaluar particiones por cliente y por tiempo; incorporar características disponibles
antes de la devolución; revisar calibración y costos de falsas alertas; estudiar errores
por subgrupo; validar estabilidad de clústeres y probar la solución con datos reales.

## 7. Publicación y evidencias por completar

- URL del repositorio público: **PENDIENTE**.
- URL real del dashboard: **PENDIENTE**.
- Commit de la versión presentada: **PENDIENTE**.
- Fecha y evidencia de prueba desde la URL pública: **PENDIENTE**.
- Figuras del dashboard publicado y revisión final de referencias: **PENDIENTE**.

## Referencias

1. Guía técnica del Proyecto Integrador de Ciencia de Datos e Inteligencia Artificial,
   Universidad de La Salle, periodo octubre–noviembre de 2026, documento proporcionado.
2. Bayona Rojas, J. J. Notebooks de Fase 1, Fase 2 y Fase 3 y artefactos proporcionados.
3. Dataset: https://www.kaggle.com/datasets/datascikhan/e-commerce-sales-and-customer-analytics
4. Streamlit Community Cloud: https://docs.streamlit.io/deploy/streamlit-community-cloud
5. Guía de despliegue: https://docs.streamlit.io/deploy/streamlit-community-cloud/deploy-your-app/deploy
'''
ruta.parent.mkdir(parents=True, exist_ok=True)
ruta.write_text(contenido, encoding="utf-8")
print("Creado:", ruta)

Creado: /workspace/scratch/95ce581e477a/proyecto_fase4/reports/informe_final_borrador.md


In [18]:
ruta = PROYECTO / "reports/guion_sustentacion.md"
contenido = r'''# Guion de sustentación — 12 minutos

| Tiempo | Contenido | Qué mostrar |
|---|---|---|
| 0:00–1:00 | Problema, objetivo y unidad de análisis | Un pedido por fila; alcance exploratorio |
| 1:00–2:15 | Fuente y calidad de datos | Datos simulados, 138.116 pedidos y alcance USD |
| 2:15–3:30 | Preparación y prevención de fuga | Variables excluidas y división 70/15/15 |
| 3:30–5:00 | Modelos y resultados | Regresión R² 0,382; clasificación AUC 0,561 y precisión 8,4 % |
| 5:00–6:00 | Segmentación | K-Means K=4 frente a DBSCAN; perfiles de pedidos |
| 6:00–7:30 | Demostración de filtros y KPI | Abrir URL, cambiar periodo y canal; explicar denominadores |
| 7:30–9:00 | Demostración de predicción | Un pedido USD; interpretar error y puntuación exploratoria |
| 9:00–10:00 | Demostración de clústeres | Mapa PCA, filtro de perfil y promedios |
| 10:00–11:00 | Límites y trabajo futuro | Datos simulados, clientes compartidos y evaluación temporal |
| 11:00–12:00 | Conclusiones y reproducibilidad | Repo público, dependencias y propuestas de piloto |

## Preparar la demostración

Abre la URL pública antes de comenzar. Usa primero USD, un periodo amplio y un canal.
En el formulario prueba 35 años, 5 unidades y una fecha del periodo 2021–2025.
Explica que el clasificador usa el umbral ya seleccionado, sin reajustarlo en prueba.
Muestra el mapa, selecciona un perfil y comenta sus promedios.
Conserva el comando local `python -m streamlit run dashboard/app.py` como respaldo.
La entrega debe incluir además la demostración desde la URL desplegada.

## Preguntas que debes poder responder

1. **¿Por qué no sumar todas las ventas?** Los importes usan monedas distintas;
   sumar EUR, USD e INR sin conversión carece de una unidad monetaria común.
2. **¿Por qué retirar puntos y datos de entrega?** En los datos se relacionaban con
   el resultado final y podían revelar la devolución o no existir al predecir.
3. **¿La puntuación 0,50 significa 50 % de probabilidad real?** No se ha comprobado
   calibración; se muestra como puntuación del clasificador balanceado.
4. **¿Por qué la exactitud no basta?** Hay desbalance; un modelo puede acertar la
   clase frecuente sin detectar devoluciones. También hay que revisar precisión,
   recall, F1, AUC y Average Precision.
5. **¿Cómo se eligió el umbral?** Maximizando F1 en validación; prueba se reservó
   para la evaluación final. Se mantuvo 0,491138 en el dashboard.
6. **¿Los grupos representan tipos permanentes de cliente?** Describen pedidos;
   un cliente puede generar pedidos que caigan en varios grupos.
7. **¿Por qué K=4?** Por la combinación del análisis del codo y la interpretación;
   debe reconocerse que la silueta no necesariamente tiene su máximo en K=4.
8. **¿PCA cambia los clústeres?** No: representa en 2D los datos transformados;
   las etiquetas provienen de K-Means en sus tres características originales.
9. **¿Hay impacto de negocio demostrado?** No. Se proponen pilotos con datos reales.
10. **¿Cómo generaliza a clientes nuevos?** La partición por pedido no resuelve esa
    pregunta; falta evaluación separada por cliente y por tiempo.
'''
ruta.parent.mkdir(parents=True, exist_ok=True)
ruta.write_text(contenido, encoding="utf-8")
print("Creado:", ruta)

Creado: /workspace/scratch/95ce581e477a/proyecto_fase4/reports/guion_sustentacion.md


In [19]:
ruta = PROYECTO / "reports/checklist_fase4.md"
contenido = r'''# Checklist de Fase 4

- [x] Filtros por fechas, categorías y variables numéricas.
- [x] KPI comerciales y métricas técnicas con denominadores definidos.
- [x] Predicción interactiva con pipelines guardados.
- [x] Umbral de clasificación conservado y desempeño limitado explicado.
- [x] Mapa PCA interactivo y perfiles promedio de K-Means.
- [x] Control de monedas: regresión y segmentos solo USD.
- [x] Código organizado en data, notebooks, src y dashboard.
- [x] Versiones fijadas para el dashboard y la inferencia.
- [x] Reproducción de métricas de prueba y de etiquetas exportadas.
- [x] Borrador del informe técnico y guion de 12 minutos.
- [ ] Ejecutar el notebook y guardar la versión ejecutada para la entrega.
- [ ] Publicar el repositorio en GitHub con acceso público.
- [ ] Desplegar en Streamlit Community Cloud y verificar el acceso.
- [ ] Probar predicción, filtros y clústeres desde la URL real.
- [ ] Registrar URL, commit y evidencia del despliegue en informe y README.
- [ ] Revisar el borrador y exportar el informe definitivo a PDF.
- [ ] Ensayar 12 minutos de exposición y 5 de preguntas.
'''
ruta.parent.mkdir(parents=True, exist_ok=True)
ruta.write_text(contenido, encoding="utf-8")
print("Creado:", ruta)

Creado: /workspace/scratch/95ce581e477a/proyecto_fase4/reports/checklist_fase4.md


## 11. Ejecutar la aplicación y descargar el proyecto

**En tu computador**, abre una terminal en la raíz del proyecto y ejecuta:

```bash
python -m pip install -r requirements.txt
python -m streamlit run dashboard/app.py
```

Abre la dirección que indica Streamlit, normalmente `http://localhost:8501`.
En Colab, la dirección `localhost` pertenece al servidor remoto. Para abrir y compartir
el dashboard usa la publicación del paso siguiente; no hace falta configurar un túnel.

La celda siguiente genera un ZIP para descargar los archivos. **Descomprímelo antes
de subirlo a GitHub.** Descarga también la versión ejecutada de este notebook desde
**Archivo → Descargar → Descargar .ipynb** y agrégala a `notebooks/`. Si subiste solo
el ZIP de Fase 3, añade tus notebooks anteriores al repositorio para completar la evidencia.

In [20]:
DESTINO_ZIP = PROYECTO.parent / "proyecto_fase4_para_github.zip"
with zipfile.ZipFile(DESTINO_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as entrega:
    for f in sorted(PROYECTO.rglob("*")):
        relativa = f.relative_to(PROYECTO)
        if not f.is_file() or any(p in {"__pycache__", ".ipynb_checkpoints", "artefactos_fase3", ".venv"} for p in relativa.parts):
            continue
        entrega.write(f, arcname=relativa)
print("Creado:", DESTINO_ZIP)
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(DESTINO_ZIP))

Creado: /workspace/scratch/95ce581e477a/proyecto_fase4_para_github.zip


## 12. Publicar: GitHub + Streamlit Community Cloud

**Sí se usa GitHub:** allí queda el repositorio que exige la asignatura. Streamlit
Community Cloud toma sus archivos y ejecuta el dashboard en una URL pública.
El servicio permite crear, desplegar y administrar apps sin costo según su documentación.
GitHub Pages publica archivos estáticos y no ejecuta el servidor Python de esta app.

1. En [GitHub](https://github.com/new), crea un repositorio público, por ejemplo
   `proyecto-ciencia-datos-ecommerce`.
2. Descomprime el ZIP y sube sus archivos y carpetas con **Add file → Upload files**.
   El contenido debe quedar en la raíz del repo: revisa que se vean `requirements.txt`
   y `dashboard/app.py`. Conserva `data/`, `models/`, `src/`, `notebooks/` y `reports/`.
3. Entra en [Streamlit Community Cloud](https://share.streamlit.io/), inicia sesión
   y conecta tu cuenta de GitHub.
4. Pulsa **Create app**, indica que ya tienes una aplicación y completa:

   | Campo | Selección |
   |---|---|
   | Repository | tu usuario / proyecto-ciencia-datos-ecommerce |
   | Branch | main, o la rama real del proyecto |
   | Main file path | dashboard/app.py |
   | Advanced settings → Python version | 3.12 |

5. Pulsa **Deploy**. Espera la instalación y abre la URL que Streamlit te entregue.
6. Prueba los cuatro módulos y el acceso en una ventana privada. Registra la URL real.
7. Añade las URL de app y repositorio al informe, a este notebook y al README.

**Errores frecuentes:** `ModuleNotFoundError` → revisar `requirements.txt` y `src/`;
`FileNotFoundError` → revisar las carpetas y mayúsculas;
error de versión de scikit-learn → usar exactamente `1.6.1`;
archivo de entrada no encontrado → no subir solamente el ZIP ni una carpeta contenedora
sin ajustar `Main file path`.

La documentación oficial describe [el servicio](https://docs.streamlit.io/deploy/streamlit-community-cloud),
[el despliegue](https://docs.streamlit.io/deploy/streamlit-community-cloud/deploy-your-app/deploy)
y [las dependencias](https://docs.streamlit.io/deploy/streamlit-community-cloud/deploy-your-app/app-dependencies).
GitHub explica las limitaciones de [GitHub Pages](https://docs.github.com/en/pages/getting-started-with-github-pages/creating-a-github-pages-site).

## 13. Registrar la publicación y cerrar la entrega

Completa la celda únicamente después de obtener y comprobar las URL. El campo
`verificado_en_navegador` lo marcas tú después de probar filtros, predicción y mapa
desde la dirección publicada. El código registra tu comprobación; no la realiza por ti.
Después de completarla, vuelve a ejecutar la celda del ZIP del paso 11 para incluir
los enlaces y el registro actualizado en tu descarga.

Guarda el notebook ejecutado, completa el informe, exporta su PDF y ensaya los 12
minutos de exposición. La app y el repo deben abrirse desde los enlaces que entregarás.

In [21]:
URL_REPOSITORIO = ""   # Pega aquí la URL real de tu repositorio público.
URL_DASHBOARD = ""     # Pega aquí la URL real que te entregó Streamlit.
COMMIT_ENTREGADO = ""  # Copia el identificador de la versión presentada.
VERIFICADO_EN_NAVEGADOR = False  # Cambia a True tras probar la app publicada.

registro = {
    "repositorio": URL_REPOSITORIO or None,
    "dashboard": URL_DASHBOARD or None,
    "commit": COMMIT_ENTREGADO or None,
    "verificado_en_navegador": VERIFICADO_EN_NAVEGADOR,
    "estado": "pendiente de publicación o verificación",
}
if URL_REPOSITORIO and URL_DASHBOARD and COMMIT_ENTREGADO and VERIFICADO_EN_NAVEGADOR:
    registro["estado"] = "publicación verificada por la autora"
(PROYECTO / "reports/publicacion.json").write_text(json.dumps(registro, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(registro, ensure_ascii=False, indent=2))

{
  "repositorio": null,
  "dashboard": null,
  "commit": null,
  "verificado_en_navegador": false,
  "estado": "pendiente de publicación o verificación"
}


## Conclusiones de Fase 4

La implementación integra los modelos seleccionados y los perfiles en una interfaz
con filtros y controles de entrada. Mantiene las transformaciones originales, el
alcance monetario y el umbral de clasificación. La evaluación se distingue de la
consulta descriptiva para evitar atribuir métricas de prueba a un filtro del dashboard.

La regresión conserva un error relevante y la clasificación tiene capacidad limitada.
Los segmentos permiten formular hipótesis, cuya efectividad requerirá un piloto.
La fase queda lista para su publicación; **solo se considera desplegada cuando exista
una URL real comprobada**. El informe y la sustentación deben reflejar ese estado.